# Prompt Engineering Assignment

## Objective

This notebook demonstrates how Large Language Models (LLMs) can be used to convert business rules into test cases.

The notebook compares:

1. GPT-4o-mini (OpenAI)
2. Llama 3.3 70B (Open Source via Groq)

Both models receive the same business rule and prompt.

The outputs are evaluated on:

- Rule interpretation
- Boundary identification
- Test case generation
- Output formatting

Business Rule:

IF FICO > 750
AND FICO <= 900
AND NO_INQ >= 2
AND NO_INQ <= 99

THEN

DECISION_CD = Declined
DECISION_DESC = INQUIRIES

## Step 1: Import Libraries

The required libraries are imported.

Purpose:

- Load API keys securely
- Connect to OpenAI
- Connect to Groq
- Store results in DataFrames
- Compare outputs

In [16]:
# Data manipulation
import pandas as pd

# Environment variable handling
import os

# Load .env variables
from dotenv import load_dotenv

# OpenAI client
from openai import OpenAI

# Groq client
from groq import Groq

## Step 2: Load API Keys

API keys are stored in a .env file.

This prevents secrets from being exposed in GitHub.

In [17]:
# Load variables from .env
load_dotenv()

# Retrieve API keys
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

# Validate keys were loaded
print("OpenAI Key Loaded:", openai_key is not None)
print("Groq Key Loaded:", groq_key is not None)

OpenAI Key Loaded: True
Groq Key Loaded: True


## Step 3: Define Business Rule

This rule will be provided to both models.

The objective is to determine:

- Minimum valid values
- Middle valid values
- Maximum valid values

and generate valid test cases.

In [18]:
RULE = """
IF FICO > 750 and FICO <= 900
AND NO_INQ >= 2 and NO_INQ <= 99

THEN

DECISION_CD = Declined
DECISION_DESC = INQUIRIES
"""

print(RULE)


IF FICO > 750 and FICO <= 900
AND NO_INQ >= 2 and NO_INQ <= 99

THEN

DECISION_CD = Declined
DECISION_DESC = INQUIRIES



## Step 4: Create Prompt

Prompt engineering is the primary focus of this assignment.

The prompt instructs the model to:

1. Identify variable ranges
2. Determine Min / Mid / Max values
3. Generate test cases
4. Follow a specific output format

In [19]:
PROMPT_V2 = f"""
You are a decision rule parser.

Rule:

{RULE}

Determine:

1. Minimum valid integer value
2. Middle valid integer value
3. Maximum valid integer value

For each variable.

Generate every possible combination.

Use only:

- Minimum
- Middle
- Maximum

Return exactly 9 tuples.

Do not explain.

Do not describe variables.

Do not provide headings.

Do not provide bullet points.

Output format:

(FICO, NO_INQ, DECISION_CD, DECISION_DESC)

Return only tuples.
"""

## Step 5: Run Experiment Using OpenAI

The prompt is sent to GPT-4o-mini.

The output is captured for later comparison.

In [20]:
# Create OpenAI client
client = OpenAI(api_key=openai_key)

# Submit prompt
response_openai = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": PROMPT_V2
        }
    ]
)

# Extract response text
openai_output = response_openai.choices[0].message.content

print(openai_output)

(751, 2, Declined, INQUIRIES)  
(751, 50, Declined, INQUIRIES)  
(751, 99, Declined, INQUIRIES)  
(850, 2, Declined, INQUIRIES)  
(850, 50, Declined, INQUIRIES)  
(850, 99, Declined, INQUIRIES)  
(900, 2, Declined, INQUIRIES)  
(900, 50, Declined, INQUIRIES)  
(900, 99, Declined, INQUIRIES)  


## Step 6: Run Experiment Using Open Source Model

The same prompt is submitted to Llama 3.3 70B through Groq.

Using the same prompt allows a fair comparison.

In [28]:
# Create Groq client
groq_client = Groq(api_key=groq_key)
model="qwen/qwen3-32b"
# Submit prompt
response_groq = groq_client.chat.completions.create(
    model=model,
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": PROMPT_V2
        }
    ]
)

# Extract response text
groq_output = response_groq.choices[0].message.content

print(groq_output)

<think>
Okay, let's tackle this problem step by step. First, I need to understand the rules given. The decision rule says that if the FICO score is greater than 750 and less than or equal to 900, and the number of inquiries (NO_INQ) is between 2 and 99 inclusive, then the decision code is Declined with the description INQUIRIES.

So, the variables here are FICO and NO_INQ. For each variable, I need to find the minimum, middle, and maximum valid integer values. Then, generate all possible combinations of these values, resulting in 9 tuples. Each tuple should include the FICO value, NO_INQ value, DECISION_CD, and DECISION_DESC.

Starting with FICO: The valid range is greater than 750 and up to 900. So the minimum valid integer is 751. The maximum is 900. The middle value would be the average of 751 and 900. Let me calculate that. (751 + 900)/2 = 1651/2 = 825.5. Since we need an integer, it's either 825 or 826. But since the range is inclusive, maybe we take the floor or ceiling. Wait, th

## Step 7: Compare Outputs

Outputs from both models are displayed side-by-side.

This helps identify:

- Formatting differences
- Reasoning differences
- Completeness of test cases

In [32]:
print("="*80)
print("OpenAI GPT-4o-mini")
print("="*80)
print(openai_output)

print("\n\n")

print("="*80)
print("Groq "+model)
print("="*80)
print(groq_output)

OpenAI GPT-4o-mini
(751, 2, Declined, INQUIRIES)  
(751, 50, Declined, INQUIRIES)  
(751, 99, Declined, INQUIRIES)  
(850, 2, Declined, INQUIRIES)  
(850, 50, Declined, INQUIRIES)  
(850, 99, Declined, INQUIRIES)  
(900, 2, Declined, INQUIRIES)  
(900, 50, Declined, INQUIRIES)  
(900, 99, Declined, INQUIRIES)  



Groq qwen/qwen3-32b
<think>
Okay, let's tackle this problem step by step. First, I need to understand the rules given. The decision rule says that if the FICO score is greater than 750 and less than or equal to 900, and the number of inquiries (NO_INQ) is between 2 and 99 inclusive, then the decision code is Declined with the description INQUIRIES.

So, the variables here are FICO and NO_INQ. For each variable, I need to find the minimum, middle, and maximum valid integer values. Then, generate all possible combinations of these values, resulting in 9 tuples. Each tuple should include the FICO value, NO_INQ value, DECISION_CD, and DECISION_DESC.

Starting with FICO: The valid